# Base-Model Complementarity Analysis

This notebook documents aggregate SVM and IndoBERT complementarity across the repeated seeds. It compares joint correctness and error overlap against silver and manual labels while excluding all candidate-level qualitative-error material from the restricted release copy.


## Release Boundary

This restricted code-inspection copy retains aggregate complementarity logic but does not include Banjarbaru review data, row-level predictions, manual-audit records, candidate rankings, or qualitative-error worksheets. The restricted path placeholder documents the authorized environment and is not an executable public-data contract.


### Configure aggregate base-model complementarity analysis

Defines the restricted project-root placeholder, the silver-label prediction source, and the approved aggregate manual-label contingency source. Candidate records, review text, individual audit labels, and private audit-prediction files are outside this release copy.

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd


PROJECT_ROOT = Path('RESTRICTED_PROJECT_ROOT')
ROUND1_DIR = PROJECT_ROOT / 'journal_round1'
R1_08_DIR = ROUND1_DIR / 'artifacts' / 'r1_08_repeated_seed_evaluation_v1'
HUMAN_DIR = ROUND1_DIR / 'reports' / 'human_label_audit' / 'r1_08_repeated_seed'
OUTPUT_DIR = ROUND1_DIR / 'reports' / 'base_model_complementarity' / 'r1_11'

SILVER_PREDICTIONS_PATH = R1_08_DIR / 'locked_test_predictions_per_model_long_with_silver.csv'
HUMAN_COMPLEMENTARITY_PATH = HUMAN_DIR / 'human_label_base_model_correctness_contingency_per_seed.csv'

EXPECTED_SEEDS = [42, 52, 62]
BASE_MODELS = [('svm', 'SVM'), ('indobert', 'IndoBERT')]

for path in [SILVER_PREDICTIONS_PATH, HUMAN_COMPLEMENTARITY_PATH]:
    if not path.is_file():
        raise FileNotFoundError(path)

print('Restricted project root placeholder:', PROJECT_ROOT)
print('Aggregate output directory:', OUTPUT_DIR)

### Define aggregate complementarity utilities

Provides helpers that turn paired SVM and IndoBERT predictions into per-seed correctness counts, disagreement rates, and error correlations. These helpers return aggregate statistics only.

In [ ]:
def phi_error_correlation(svm_error: np.ndarray, indobert_error: np.ndarray) -> float:
    left = np.asarray(svm_error, dtype=float)
    right = np.asarray(indobert_error, dtype=float)
    if left.std(ddof=0) == 0 or right.std(ddof=0) == 0:
        return float('nan')
    return float(np.corrcoef(left, right)[0, 1])


def build_base_wide(frame: pd.DataFrame, target_column: str, target_name: str, expected_n: int) -> pd.DataFrame:
    required = {'seed', 'row_id', 'model_key', 'prediction', target_column}
    missing = sorted(required.difference(frame.columns))
    if missing:
        raise ValueError(f'{target_name}: missing columns {missing}.')
    base = frame.loc[frame['model_key'].isin([key for key, _ in BASE_MODELS])].copy()
    if sorted(base['seed'].unique().tolist()) != EXPECTED_SEEDS:
        raise ValueError(f'{target_name}: unexpected seed set.')
    if base.duplicated(['seed', 'row_id', 'model_key']).any():
        raise ValueError(f'{target_name}: duplicate seed-row-model key.')
    target_n = base.groupby(['seed', 'row_id'])[target_column].nunique()
    if not target_n.eq(1).all():
        raise ValueError(f'{target_name}: target labels differ between base-model rows.')

    prediction = base.pivot(index=['seed', 'row_id', target_column], columns='model_key', values='prediction').reset_index()
    if len(prediction) != expected_n * len(EXPECTED_SEEDS):
        raise ValueError(f'{target_name}: unexpected paired prediction count.')
    if prediction[[key for key, _ in BASE_MODELS]].isna().any().any():
        raise ValueError(f'{target_name}: base-model prediction is missing.')
    prediction = prediction.rename(columns={target_column: 'target'})
    prediction['svm_correct'] = prediction['svm'].eq(prediction['target'])
    prediction['indobert_correct'] = prediction['indobert'].eq(prediction['target'])
    prediction['svm_error'] = ~prediction['svm_correct']
    prediction['indobert_error'] = ~prediction['indobert_correct']
    prediction['hard_prediction_disagreement'] = prediction['svm'].ne(prediction['indobert'])
    prediction['reference_target'] = target_name
    return prediction.sort_values(['seed', 'row_id']).reset_index(drop=True)


def complementarity_per_seed(wide: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for seed, group in wide.groupby('seed', sort=True):
        svm_correct = group['svm_correct'].to_numpy(bool)
        indobert_correct = group['indobert_correct'].to_numpy(bool)
        svm_error = ~svm_correct
        indobert_error = ~indobert_correct
        rows.append({
            'reference_target': group['reference_target'].iloc[0], 'seed': int(seed), 'n': int(len(group)),
            'both_correct_n': int(np.sum(svm_correct & indobert_correct)),
            'svm_only_correct_n': int(np.sum(svm_correct & ~indobert_correct)),
            'indobert_only_correct_n': int(np.sum(~svm_correct & indobert_correct)),
            'both_incorrect_n': int(np.sum(svm_error & indobert_error)),
            'hard_prediction_disagreement_n': int(group['hard_prediction_disagreement'].sum()),
            'hard_prediction_disagreement_rate': float(group['hard_prediction_disagreement'].mean()),
            'svm_error_rate': float(svm_error.mean()), 'indobert_error_rate': float(indobert_error.mean()),
            'phi_error_correlation': phi_error_correlation(svm_error, indobert_error),
        })
    return pd.DataFrame(rows).sort_values(['reference_target', 'seed']).reset_index(drop=True)

### Combine silver-label and manual-label aggregate complementarity evidence

Calculates complementarity against locked-test silver labels and reads the approved aggregate contingency table for the manual audit. It validates the shared three-seed contract before summarising both evaluation targets.

In [ ]:
silver_long = pd.read_csv(SILVER_PREDICTIONS_PATH)
silver_wide = build_base_wide(silver_long, 'y_silver', 'locked_test_silver', expected_n=1172)
silver_complementarity = complementarity_per_seed(silver_wide)

manual_complementarity = pd.read_csv(HUMAN_COMPLEMENTARITY_PATH)
required_manual_columns = {
    'reference_target', 'seed', 'n', 'both_correct_n', 'svm_only_correct_n', 'indobert_only_correct_n',
    'both_incorrect_n', 'hard_prediction_disagreement_n', 'hard_prediction_disagreement_rate',
    'svm_error_rate', 'indobert_error_rate', 'phi_error_correlation',
}
missing_manual_columns = sorted(required_manual_columns.difference(manual_complementarity.columns))
if missing_manual_columns:
    raise ValueError(f'Aggregate manual-label contingency is missing columns: {missing_manual_columns}.')
if manual_complementarity['reference_target'].ne('human_audit_manual').any():
    raise ValueError('Aggregate manual-label contingency has an unexpected reference target.')
if sorted(manual_complementarity['seed'].astype(int).unique().tolist()) != EXPECTED_SEEDS:
    raise ValueError('Aggregate manual-label contingency has an unexpected seed set.')
if len(manual_complementarity) != len(EXPECTED_SEEDS):
    raise ValueError('Aggregate manual-label contingency must contain one row per seed.')

complementarity = pd.concat([silver_complementarity, manual_complementarity], ignore_index=True)

summary_columns = [
    'both_correct_n', 'svm_only_correct_n', 'indobert_only_correct_n', 'both_incorrect_n',
    'hard_prediction_disagreement_n', 'hard_prediction_disagreement_rate',
    'svm_error_rate', 'indobert_error_rate', 'phi_error_correlation',
]
summary = complementarity.groupby('reference_target', as_index=False).agg(seed_n=('seed', 'nunique'))
for column in summary_columns:
    aggregate = complementarity.groupby('reference_target')[column].agg(['mean', 'std']).reset_index()
    aggregate = aggregate.rename(columns={'mean': f'{column}_mean', 'std': f'{column}_sample_sd'})
    summary = summary.merge(aggregate, on='reference_target', how='left', validate='one_to_one')

if len(complementarity) != 6 or not complementarity.groupby('reference_target')['seed'].nunique().eq(3).all():
    raise RuntimeError('Complementarity output is incomplete.')

print(complementarity.to_string(index=False))
print(summary.to_string(index=False))

### Write aggregate complementarity tables

Exports only per-seed and summary-level contingency statistics. The release copy does not write candidate worksheets, per-record predictions, audit records, run-state files, or generated manifests.

In [ ]:
aggregate_exports = {
    'base_model_correctness_contingency_per_seed.csv': complementarity,
    'base_model_complementarity_summary.csv': summary,
    'base_model_error_correlation_per_seed.csv': complementarity.loc[:, [
        'reference_target', 'seed', 'n', 'svm_error_rate', 'indobert_error_rate', 'phi_error_correlation',
    ]].copy(),
}

for filename, frame in aggregate_exports.items():
    frame.to_csv(OUTPUT_DIR / filename, index=False, encoding='utf-8')
    print('Wrote aggregate complementarity file:', filename)

print('Base-model complementarity exported as approved aggregate outputs only.')